# July 2021 floods: starter notebook

This notebook does two things:
1. **Connects** you to the challenge data in the cloud. Nothing is downloaded, and no login is needed: the data is public.
2. **Shows what data is available** (EMS flood maps, EFAS flood hazard, GHS-OBAT buildings, GHS-POP population for DE, BE, NL).

Click **Run all**. Works in Colab and on a laptop.

<details>
<summary><b>Not using Colab? (VS Code, Jupyter, any Python)</b></summary>

1. `pip install gcsfs pyogrio rasterio geopandas shapely ipykernel`
   (on Windows, if that fails: `conda install -c conda-forge geopandas rasterio pyogrio gcsfs`)
2. VS Code: install the Python and Jupyter extensions and pick your environment as kernel. If the packages are already installed there, you can skip the `pip install` cell below. After installing, restart the kernel.

**Problems:** *connection or timeout errors* → check your internet connection and retry · *`No such file` on a path* → run the Explore cell to see the exact file names · *`ModuleNotFoundError`* → the kernel and your terminal use different Pythons: install with `%pip install gcsfs` inside the notebook, then restart the kernel.

</details>

## 1. Setup

In [1]:
%%capture
%pip install -q gcsfs pyogrio rasterio geopandas shapely

In [2]:
import os, re, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings("ignore")

BUCKET = "datasets_challenge"
FOLDER = "working_data"

## 2. Connect

In [3]:
import gcsfs

# The bucket is public: no login needed.
os.environ["GS_NO_SIGN_REQUEST"] = "YES"      # lets GDAL (rasterio, pyogrio) read it anonymously
GDAL_PREFIX = "/vsigs/"
ROOT = f"{BUCKET}/{FOLDER}"
fs = gcsfs.GCSFileSystem(token="anon")
print("Connected:", [Path(p).name for p in fs.ls(ROOT)])

Connected: ['cems-efas', 'ems_be', 'ems_de', 'ems_nl', 'ghs_pop', 'obat_be', 'obat_de', 'obat_nl']


## 3. Explore

In [6]:
for folder in ["cems-efas", "ghs_pop", "obat_be", "obat_de", "obat_nl", "ems_be", "ems_de", "ems_nl"]:
    print(f"\n{folder}")
    files = fs.find(f"{ROOT}/{folder}", detail=True)
    if folder.startswith("ems_"):
        # vector products: one line per product directory
        dirs = {}
        for p, info in sorted(files.items()):
            d = dirs.setdefault(Path(p).parent.name, [0, 0])
            d[0] += p.lower().endswith(".shp"); d[1] += int(info["size"])
        for name, (n_shp, size) in dirs.items(): print("  ", name, f"{n_shp} shp, {size/1e6:,.1f} MB")
    else:
        for p, info in sorted(files.items()):
            if p.lower().endswith((".tif", ".tiff", ".gpkg")): print("  ", Path(p).name, f"{int(info['size'])/1e6:,.0f} MB")

# EMS: one row per area (aoi), type (DEL/GRA) and version, with its layers
EMS_RE = re.compile(r"(EMSR\d+)_(AOI\d+)_([A-Z]+)_(PRODUCT|MONIT\d*)_(\w+?)_r\d+_v\d+", re.I)
rows = []
for cc in ("de", "be", "nl"):
    for p in fs.find(f"{ROOT}/ems_{cc}"):
        m = EMS_RE.search(Path(p).name) if p.lower().endswith(".shp") else None
        if m: rows.append(dict(cc=cc, aoi=m[2], ptype=m[3].upper(), ver=m[4].upper(), layer=m[5]))
display(pd.DataFrame(rows).groupby(["cc", "aoi", "ptype", "ver"]).layer.apply(lambda s: ", ".join(sorted(s))).to_frame())


cems-efas
   Europe_RP100_filled_depth.tif 323 MB
   Europe_RP10_filled_depth.tif 272 MB
   Europe_RP500_filled_depth.tif 350 MB

ghs_pop
   GHS_POP_E2020_GLOBE_R2023A_4326_30ss_V1_0.tif 383 MB
   GHS_POP_E2025_GLOBE_R2023A_4326_30ss_V1_0.tif 384 MB

obat_be
   GHS_OBAT_GPKG_BEL_E2020_R2024A_V1_0.gpkg 1,433 MB

obat_de
   GHS_OBAT_GPKG_DEU_E2020_R2024A_V1_0.gpkg 8,579 MB

obat_nl
   GHS_OBAT_GPKG_NLD_E2020_R2024A_V1_0.gpkg 2,229 MB

ems_be
   EMSR518_AOI01_DEL_MONIT01_r1_VECTORS_v1_vector 9 shp, 65.4 MB
   EMSR518_AOI01_DEL_PRODUCT_r1_VECTORS_v1_vector 10 shp, 76.3 MB
   EMSR518_AOI02_DEL_MONIT01_r1_VECTORS_v1_vector 12 shp, 110.3 MB
   EMSR518_AOI02_DEL_PRODUCT_r1_VECTORS_v1_vector 10 shp, 138.3 MB

ems_de
   EMSR517_AOI01_DEL_MONIT01_r1_RTP01_v1_vector 11 shp, 21.3 MB
   EMSR517_AOI01_DEL_PRODUCT_r1_RTP01_v1_vector 10 shp, 21.3 MB
   EMSR517_AOI03_DEL_MONIT01_r1_RTP01_v1_vector 10 shp, 6.9 MB
   EMSR517_AOI03_DEL_PRODUCT_r1_RTP01_v1_vector 10 shp, 5.2 MB
   EMSR517_AOI04_DEL_MONIT01

layer
cc aoi   ptype ver                                                       
be AOI01 DEL   MONIT01  areaOfInterestA, builtUpA, facilitiesL, hydrog...
               PRODUCT  areaOfInterestA, builtUpA, facilitiesL, hydrog...
   AOI02 DEL   MONIT01  areaOfInterestA, builtUpA, facilitiesA, facili...
               PRODUCT  areaOfInterestA, facilitiesL, hydrographyA, hy...
de AOI01 DEL   MONIT01  areaOfInterestA, builtUpA, facilitiesA, facili...
               PRODUCT  areaOfInterestA, builtUpA, facilitiesL, hydrog...
   AOI03 DEL   MONIT01  areaOfInterestA, builtUpA, facilitiesA, facili...
               PRODUCT  areaOfInterestA, builtUpA, facilitiesA, facili...
   AOI04 DEL   MONIT01  areaOfInterestA, builtUpA, facilitiesA, facili...
               PRODUCT  areaOfInterestA, builtUpA, facilitiesL, hydrog...
   AOI05 DEL   MONIT01  areaOfInterestA, builtUpA, facilitiesL, hydrog...
               PRODUCT  areaOfInterestA, builtUpA, facilitiesL, hydrog...
   AOI06 DEL   MONIT01  areaOfInterestA, builtUpA, facilitiesL, hydrog...
               PRODUCT  areaOfInterestA, builtUpA, facilitiesL, hydrog...
   AOI07 GRA   PRODUCT  areaOfInterestA, builtUpP, facilitiesA, hydrog...
   AOI08 GRA   PRODUCT  areaOfInterestA, builtUpP, facilitiesA, hydrog...
   AOI09 GRA   PRODUCT  areaOfInterestA, builtUpP, facilitiesA, hydrog...
   AOI10 GRA   PRODUCT  areaOfInterestA, builtUpP, facilitiesA, facili...
   AOI11 GRA   PRODUCT  areaOfInterestA, builtUpP, facilitiesA, hydrog...
   AOI14 GRA   PRODUCT  areaOfInterestA, builtUpP, hydrographyA, hydro...
   AOI15 GRA   MONIT01  areaOfInterestA, builtUpP, facilitiesA, hydrog...
               PRODUCT  areaOfInterestA, builtUpP, facilitiesA, hydrog...
   AOI16 GRA   PRODUCT  areaOfInterestA, builtUpP, facilitiesA, hydrog...
   AOI17 GRA   PRODUCT  areaOfInterestA, builtUpP, facilitiesA, hydrog...
   AOI18 GRA   PRODUCT  areaOfInterestA, builtUpP, facilitiesA, hydrog...
   AOI20 GRA   PRODUCT  areaOfInterestA, facilitiesA, hydrographyA, hy...
nl AOI01 DEL   PRODUCT  areaOfInterestA, builtUpA, facilitiesA, hydrog...

## 4. Your analysis
Read data straight from the bucket, no login needed: list files with `fs` (gcsfs), or open them with GDAL-based libraries using the `GDAL_PREFIX` path prefix, for example `rasterio.open(f"{GDAL_PREFIX}{path}")`. The bucket is read-only, so save results to the runtime disk and download them from the file browser.

In [5]:
# your code